# Question 2: Transition-Based Dependency Parser

This notebook builds an arc-standard transition-based dependency parser trained on UD_English-EWT.

The parser keeps a **stack** (words already processed) and a **buffer** (words not yet seen), and at every step applies one of three moves:

- **SHIFT** — move the first word of the buffer onto the stack.
- **LEFT-ARC(label)** — top of stack becomes head of the second word on the stack; the second word is popped.
- **RIGHT-ARC(label)** — second word on the stack becomes head of the top word; the top word is popped.

This continues until the buffer is empty and only `ROOT` remains on the stack. A classifier trained on POS-tag features decides which move to make at each step.

**Contents:**
1. CoNLL-U file reading.
2. Oracle simulation to generate training data from gold trees.
3. Feature extraction.
4. Classifier training.
5. Parser inference loop.
6. LAS/UAS evaluation on the dev set.
7. Parsing a few example sentences.

In [1]:
import time
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

DATA_DIR = "dataset-UD_English-EWT"  # dataset folder sitting next to this notebook

## Part 1: Data Processing and Oracle Simulation

### Part 1.1 — CoNLL-U Parser

A `.conllu` file has one word per line, tab-separated. The columns used here:

- column 1: word id (1-indexed within the sentence)
- column 2: the word form
- column 4: universal POS tag (UPOS)
- column 7: head word id (0 = attaches to ROOT)
- column 8: dependency relation label (e.g. `nsubj`, `obj`, `det`)

Sentences are separated by blank lines, `#` lines are comments, and lines describing multiword tokens (id like `3-4`) or empty nodes (id like `3.1`) are skipped since they aren't part of the basic tree.

Dependency labels are simplified by dropping any subtype after a colon (e.g. `obl:tmod` -> `obl`), to keep the number of classes the classifier has to learn manageable.

In [2]:
def read_conllu(path):
    """Read a .conllu file and return a list of sentences.
    Each sentence is a list of token dicts: id, form, upos, head, deprel."""
    sentences = []
    tokens = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("#"):
                continue
            if line.strip() == "":
                if tokens:
                    sentences.append(tokens)
                    tokens = []
                continue
            cols = line.split("\t")
            tid = cols[0]
            if "-" in tid or "." in tid:
                # multiword token range or empty node, skip
                continue
            form, upos, head, deprel = cols[1], cols[3], cols[6], cols[7]
            deprel = deprel.split(":")[0]  # drop subtype, e.g. obl:tmod -> obl
            tokens.append({
                "id": int(tid),
                "form": form,
                "upos": upos,
                "head": int(head),
                "deprel": deprel,
            })
        if tokens:
            sentences.append(tokens)
    return sentences


train_sents = read_conllu(f"{DATA_DIR}/en_ewt-ud-train.conllu")
dev_sents = read_conllu(f"{DATA_DIR}/en_ewt-ud-dev.conllu")

print("Number of training sentences:", len(train_sents))
print("Number of dev sentences:", len(dev_sents))
print()
print("Example sentence (first 8 tokens):")
for tok in train_sents[0][:8]:
    print(tok)

Number of training sentences: 12544
Number of dev sentences: 2001

Example sentence (first 8 tokens):
{'id': 1, 'form': 'Al', 'upos': 'PROPN', 'head': 0, 'deprel': 'root'}
{'id': 2, 'form': '-', 'upos': 'PUNCT', 'head': 3, 'deprel': 'punct'}
{'id': 3, 'form': 'Zaman', 'upos': 'PROPN', 'head': 1, 'deprel': 'flat'}
{'id': 4, 'form': ':', 'upos': 'PUNCT', 'head': 7, 'deprel': 'punct'}
{'id': 5, 'form': 'American', 'upos': 'ADJ', 'head': 6, 'deprel': 'amod'}
{'id': 6, 'form': 'forces', 'upos': 'NOUN', 'head': 7, 'deprel': 'nsubj'}
{'id': 7, 'form': 'killed', 'upos': 'VERB', 'head': 1, 'deprel': 'parataxis'}
{'id': 8, 'form': 'Shaikh', 'upos': 'PROPN', 'head': 7, 'deprel': 'obj'}


### Part 1.2 — Oracle Simulator

The oracle takes a sentence with a known gold dependency tree and replays the arc-standard parsing process, picking the correct move at every step. This produces `(configuration, correct_transition)` pairs, which become the training data for the classifier.

A special `ROOT` token with id `0` sits at the bottom of the stack. At each step, with `s1` = top of stack and `s2` = second item:

- If `s1` is the gold head of `s2`, and `s2` has no children still waiting in the buffer, apply **LEFT-ARC** (attach `s2` under `s1`, remove `s2`).
- Else if `s2` is the gold head of `s1`, and `s1` has no children still waiting in the buffer, apply **RIGHT-ARC** (attach `s1` under `s2`, remove `s1`).
- Otherwise apply **SHIFT**.

The "no children waiting" check is needed because a word must not be removed from the stack before all of its own dependents have been attached to it — otherwise those dependents become unreachable.

Arc-standard can only build **projective** trees (no crossing arcs). A small fraction of sentences in the treebank are non-projective, so the oracle gets stuck on those — those sentences are skipped when building the training data.

In [3]:
ROOT = {"id": 0, "form": "ROOT", "upos": "ROOT", "head": -1, "deprel": "root"}


def oracle_transitions(sent):
    """Replay arc-standard parsing on a gold sentence.
    Returns (list of (stack, buffer, action) examples, predicted arcs)
    or None if the sentence cannot be parsed by arc-standard (non-projective)."""
    tokens = {0: ROOT}
    for t in sent:
        tokens[t["id"]] = t

    stack = [0]
    buffer = [t["id"] for t in sent]
    arcs = []
    examples = []

    def has_unattached_child(wid, buffer):
        return any(tokens[bid]["head"] == wid for bid in buffer)

    max_steps = 4 * (len(sent) + 1)  # safety cap, a normal parse needs about 2n steps
    steps = 0
    while not (len(stack) == 1 and len(buffer) == 0):
        steps += 1
        if steps > max_steps:
            return None

        if len(stack) < 2:
            action = "SHIFT"
        else:
            s1, s2 = stack[-1], stack[-2]
            if s2 != 0 and tokens[s2]["head"] == s1 and not has_unattached_child(s2, buffer):
                action = f"LEFT-ARC-{tokens[s2]['deprel']}"
            elif tokens[s1]["head"] == s2 and not has_unattached_child(s1, buffer):
                action = f"RIGHT-ARC-{tokens[s1]['deprel']}"
            elif buffer:
                action = "SHIFT"
            else:
                return None  # stuck: non-projective sentence

        examples.append((list(stack), list(buffer), action))

        if action == "SHIFT":
            stack.append(buffer.pop(0))
        elif action.startswith("LEFT-ARC"):
            label = action[len("LEFT-ARC-"):]
            s1, s2 = stack[-1], stack[-2]
            arcs.append((s1, s2, label))
            stack.pop(-2)
        else:  # RIGHT-ARC
            label = action[len("RIGHT-ARC-"):]
            s1, s2 = stack[-1], stack[-2]
            arcs.append((s2, s1, label))
            stack.pop(-1)

    return examples, arcs


# quick sanity check: does the oracle reconstruct the exact gold tree?
res = oracle_transitions(train_sents[0])
examples, arcs = res
gold = set((t["head"], t["id"]) for t in train_sents[0])
pred = set((h, d) for h, d, l in arcs)
print("Oracle reproduces gold tree exactly:", gold == pred)
print("Number of oracle steps for this sentence:", len(examples))

Oracle reproduces gold tree exactly: True
Number of oracle steps for this sentence: 58


In [4]:
# Run the oracle over every training sentence to build the training data.
all_examples = []  # each item: (sentence, stack, buffer, action)
num_ok, num_skipped = 0, 0

for sent in train_sents:
    result = oracle_transitions(sent)
    if result is None:
        num_skipped += 1
        continue
    examples, arcs = result
    gold = set((t["head"], t["id"]) for t in sent)
    pred = set((h, d) for h, d, l in arcs)
    if gold != pred:
        # arcs don't match gold exactly (can happen on rare edge cases) - skip
        num_skipped += 1
        continue
    num_ok += 1
    all_examples.extend([(sent, s, b, a) for s, b, a in examples])

print(f"Sentences successfully used for oracle training data: {num_ok}")
print(f"Sentences skipped (non-projective / oracle could not complete): {num_skipped}")
print(f"Total (configuration, transition) training examples: {len(all_examples)}")

Sentences successfully used for oracle training data: 12257
Sentences skipped (non-projective / oracle could not complete): 287
Total (configuration, transition) training examples: 392242


## Part 2: Feature Extraction and Model Training

### Part 2.1 — Feature Extractor

Features are kept to just the POS tag of:

- the word on top of the stack (`s1`)
- the second word on the stack (`s2`), if it exists
- the first word in the buffer (`b1`), if it exists
- the second word in the buffer (`b2`), if it exists

`"NONE"` is used as a placeholder when a position doesn't exist.

In [5]:
def extract_features(stack, buffer, tokens):
    """Extract POS-tag features from a parser configuration."""
    def pos(wid):
        return tokens[wid]["upos"] if wid is not None else "NONE"

    s1 = stack[-1] if len(stack) >= 1 else None
    s2 = stack[-2] if len(stack) >= 2 else None
    b1 = buffer[0] if len(buffer) >= 1 else None
    b2 = buffer[1] if len(buffer) >= 2 else None

    return {
        "s1_pos": pos(s1),
        "s2_pos": pos(s2),
        "b1_pos": pos(b1),
        "b2_pos": pos(b2),
    }


# turn every (sentence, stack, buffer, action) example into (feature dict, label)
X_dicts = []
y = []
for sent, stack, buffer, action in all_examples:
    tokens = {0: ROOT}
    for t in sent:
        tokens[t["id"]] = t
    X_dicts.append(extract_features(stack, buffer, tokens))
    y.append(action)

print("Example feature dict:", X_dicts[0], "-> label:", y[0])
print("Number of distinct transition classes (SHIFT + LEFT/RIGHT-ARC per label):", len(set(y)))

Example feature dict: {'s1_pos': 'ROOT', 's2_pos': 'NONE', 'b1_pos': 'PROPN', 'b2_pos': 'PUNCT'} -> label: SHIFT
Number of distinct transition classes (SHIFT + LEFT/RIGHT-ARC per label): 61


### Part 2.2 — Model Training

Each transition is encoded as a single class label, e.g. `SHIFT`, `LEFT-ARC-det`, `RIGHT-ARC-obj` — so one classifier predicts both the structural move and the dependency label together.

`DictVectorizer` turns the POS-tag feature dicts into one-hot vectors, and `LogisticRegression` is trained on top of that to predict the transition.

In [6]:
vec = DictVectorizer(sparse=True)
X = vec.fit_transform(X_dicts)
print("Feature matrix shape:", X.shape)

clf = LogisticRegression(max_iter=200)

t0 = time.time()
clf.fit(X, y)
t1 = time.time()

print(f"Training time: {t1 - t0:.1f} seconds")
print(f"Training accuracy (per-transition, not the same as parser accuracy): {clf.score(X, y):.4f}")

Feature matrix shape: (392242, 73)


Training time: 60.8 seconds


Training accuracy (per-transition, not the same as parser accuracy): 0.8130


## Part 3: Parser Implementation and Evaluation

### Part 3.1 — Parser

At inference time there's no gold tree to check against, so every predicted move needs to be checked for legality given the current configuration:

- `SHIFT` is legal only if the buffer is non-empty.
- `LEFT-ARC` / `RIGHT-ARC` are legal only if the stack has at least 2 words.
- `LEFT-ARC` is additionally illegal if `s2` is `ROOT` (ROOT can never be a dependent).

At each step, all transition classes are ranked by the classifier's confidence score, and the highest-scoring legal one is picked, instead of blindly applying the top prediction.

In [7]:
classes = clf.classes_


def legal_actions(stack, buffer):
    legal = set()
    if buffer:
        legal.add("SHIFT")
    if len(stack) >= 2:
        legal.add("RIGHT-ARC")
        if stack[-2] != 0:
            legal.add("LEFT-ARC")
    return legal


def action_kind(action):
    if action == "SHIFT":
        return "SHIFT"
    if action.startswith("LEFT-ARC"):
        return "LEFT-ARC"
    return "RIGHT-ARC"


def parse_sentence(sent_tokens, clf, vec):
    """Parse a sentence (list of dicts with id/form/upos) using the trained classifier.
    Returns predicted arcs as a list of (head_id, dependent_id, label)."""
    tokens = {0: ROOT}
    for t in sent_tokens:
        tokens[t["id"]] = t

    stack = [0]
    buffer = [t["id"] for t in sent_tokens]
    arcs = []

    max_steps = 4 * (len(sent_tokens) + 1)
    steps = 0
    while not (len(stack) == 1 and len(buffer) == 0):
        steps += 1
        if steps > max_steps:
            break

        legal = legal_actions(stack, buffer)
        if not legal:
            break

        feats = extract_features(stack, buffer, tokens)
        Xf = vec.transform([feats])
        scores = clf.decision_function(Xf)[0]
        order = np.argsort(-scores)

        chosen = None
        for idx in order:
            cand = classes[idx]
            if action_kind(cand) in legal:
                chosen = cand
                break
        if chosen is None:
            break

        if chosen == "SHIFT":
            stack.append(buffer.pop(0))
        elif chosen.startswith("LEFT-ARC"):
            label = chosen[len("LEFT-ARC-"):]
            s1, s2 = stack[-1], stack[-2]
            arcs.append((s1, s2, label))
            stack.pop(-2)
        else:
            label = chosen[len("RIGHT-ARC-"):]
            s1, s2 = stack[-1], stack[-2]
            arcs.append((s2, s1, label))
            stack.pop(-1)

    return arcs

### Part 3.2 — Evaluation: Labeled Attachment Score (LAS)

The trained parser is run on every sentence in the dev set, comparing the predicted head + label for each word against the gold head + label.

- **UAS** (Unlabeled Attachment Score) = fraction of words with the correct **head**.
- **LAS** (Labeled Attachment Score) = fraction of words with the correct **head AND correct label**.

LAS is stricter since it also requires the dependency relation to match.

In [8]:
total = 0
correct_unlabeled = 0
correct_labeled = 0

t0 = time.time()
for sent in dev_sents:
    pred_arcs = parse_sentence(sent, clf, vec)
    pred = {d: (h, l) for h, d, l in pred_arcs}
    for t in sent:
        total += 1
        if t["id"] in pred:
            ph, pl = pred[t["id"]]
            if ph == t["head"]:
                correct_unlabeled += 1
                if pl == t["deprel"]:
                    correct_labeled += 1
t1 = time.time()

uas = correct_unlabeled / total
las = correct_labeled / total

print(f"Evaluated on {len(dev_sents)} dev sentences ({total} words) in {t1 - t0:.1f} seconds")
print(f"UAS (Unlabeled Attachment Score): {uas:.4f}")
print(f"LAS (Labeled Attachment Score):   {las:.4f}")

Evaluated on 2001 dev sentences (25148 words) in 9.8 seconds
UAS (Unlabeled Attachment Score): 0.6714
LAS (Labeled Attachment Score):   0.5815


## Testing on Example Sentences

A few example sentences are parsed here. The parser expects a sentence as a list of `(word, POS tag)` pairs, so these are manually tagged with the same Universal POS tagset used in the training data.

In [9]:
def make_sentence(word_pos_pairs):
    return [{"id": i + 1, "form": w, "upos": p} for i, (w, p) in enumerate(word_pos_pairs)]


def print_parse(sent_tokens, arcs):
    id_to_form = {0: "ROOT"}
    for t in sent_tokens:
        id_to_form[t["id"]] = t["form"]
    head_of = {d: (h, l) for h, d, l in arcs}
    for t in sent_tokens:
        h, l = head_of.get(t["id"], ("?", "?"))
        print(f"  {t['form']:<12} <- {id_to_form.get(h, '?'):<12} ({l})")


example_sentences = {
    "The cat sat on the mat.": [
        ("The", "DET"), ("cat", "NOUN"), ("sat", "VERB"), ("on", "ADP"),
        ("the", "DET"), ("mat", "NOUN"), (".", "PUNCT"),
    ],
    "She eats a green salad.": [
        ("She", "PRON"), ("eats", "VERB"), ("a", "DET"), ("green", "ADJ"),
        ("salad", "NOUN"), (".", "PUNCT"),
    ],
    "I saw the man with a telescope.": [
        ("I", "PRON"), ("saw", "VERB"), ("the", "DET"), ("man", "NOUN"),
        ("with", "ADP"), ("a", "DET"), ("telescope", "NOUN"), (".", "PUNCT"),
    ],
}

for text, pairs in example_sentences.items():
    sent_tokens = make_sentence(pairs)
    arcs = parse_sentence(sent_tokens, clf, vec)
    print(f"Sentence: {text}")
    print_parse(sent_tokens, arcs)
    print()

Sentence: The cat sat on the mat.
  The          <- cat          (det)
  cat          <- sat          (nsubj)
  sat          <- ROOT         (root)
  on           <- mat          (case)
  the          <- mat          (det)
  mat          <- sat          (obl)
  .            <- sat          (punct)

Sentence: She eats a green salad.
  She          <- eats         (nsubj)
  eats         <- ROOT         (root)
  a            <- salad        (det)
  green        <- salad        (amod)
  salad        <- eats         (obl)
  .            <- eats         (punct)

Sentence: I saw the man with a telescope.
  I            <- saw          (nsubj)
  saw          <- ROOT         (root)
  the          <- man          (det)
  man          <- saw          (obl)
  with         <- telescope    (case)
  a            <- telescope    (det)
  telescope    <- man          (nmod)
  .            <- saw          (punct)

